### Kafka Topic Initialization

This section creates the Kafka topic used for the streaming pipeline.

The topic is created using Kafka’s **AdminClient API** and configured with a
single partition and replication factor of one, which is sufficient for a local
experimental setup.  

The function also checks whether the topic already exists to avoid errors when
the notebook is executed multiple times.

In [1]:
from confluent_kafka.admin import AdminClient, NewTopic

TOPIC_NAME = "final_run_000"

def create_kafka_topic(topic_name):
    admin_client = AdminClient({"bootstrap.servers": "localhost:9092"})
    
    # Create a single-partition topic for the local streaming experiments
    new_topic = NewTopic(topic_name, num_partitions=1, replication_factor=1)
    
    fs = admin_client.create_topics([new_topic])
    for topic, f in fs.items():
        try:
            f.result()  
            print(f" Topic '{topic}' created successfully and is ready!")
        except Exception as e:
            if "TOPIC_ALREADY_EXISTS" in str(e):
                print(f" Topic '{topic}' already exists and is ready.")
            else:
                print(f" Failed to create topic '{topic}': {e}")

create_kafka_topic(TOPIC_NAME)

 Topic 'final_run_000' created successfully and is ready!


### Environment Setup and Imports

This section imports all required libraries for the distributed streaming GMM
pipeline.

The notebook uses:
- **Kafka (confluent_kafka)** to consume streaming messages
- **Spark** for distributed processing, window alignment and warm start
- **NumPy / SciPy** for numerical operations and multivariate normal densities
- **Pandas** for collecting and displaying the final experiment summary table
- various **PySpark SQL** types and functions for schema enforcement and
  distributed feature alignment


In [2]:
"""### Environment Setup and Imports

This section imports all required libraries for the distributed streaming GMM
pipeline.

The notebook uses:
- **Kafka (confluent_kafka)** to consume streaming messages
- **Spark** for distributed processing, window alignment and warm start
- **NumPy / SciPy** for numerical operations and multivariate normal densities
- **Pandas** for collecting and displaying the final experiment summary table
- various **PySpark SQL** types and functions for schema enforcement and
  distributed feature alignment
"""

import json
import math
import time
import uuid
import random

import numpy as np
import pandas as pd
from scipy.stats import multivariate_normal
import IPython.display as display

from confluent_kafka import Consumer, KafkaError

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, DoubleType, ArrayType, IntegerType
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import GaussianMixture
from pyspark.ml.linalg import Vectors

### Spark Session Initialization and Warm Start

This section initializes the **Apache Spark session** used to process the
streaming data received from Kafka and performs a **warm start** tailored to
the GMM workload.

The code:
- creates a Spark session and sets the log level to **WARN** to reduce log noise
- generates a small dummy DataFrame with **15 synthetic feature columns** to
  match the dimensionality of the real Spotify data
- runs a `VectorAssembler` on this dummy DataFrame so Spark’s optimizer
  generates code for wide feature vectors
- fits a lightweight `GaussianMixture` model with \(K = 15\) on the dummy
  features to force MLlib to initialize its linear algebra stack and GMM code

By doing this once at startup, the first real streaming windows avoid paying the
full JVM and optimizer initialization cost, so the measured per‑window times are
dominated by the actual incremental GMM work.

In [3]:

spark = SparkSession.builder.appName("KafkaConsumer-DistributedGMM").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print("Warming up JVM and Catalyst Optimizer for 15 features...")

# Generate dummy data that matches the 15-feature shape of the Spotify data
dummy_records = [{f"feature_{i}": random.random() for i in range(15)} for _ in range(30)]
dummy_df = spark.createDataFrame(dummy_records)

# Run a VectorAssembler to force Catalyst to generate code for wide feature vectors
assembler = VectorAssembler(inputCols=dummy_df.columns, outputCol="features")
dummy_data = assembler.transform(dummy_df).select("features")

# Fit a tiny GMM to trigger MLlib and BLAS/LAPACK initialization for K=15
GaussianMixture(k=15, maxIter=1, seed=42).fit(dummy_data)

print("Warmup complete! Spark is fully optimized for the streaming run.")

26/03/23 15:02:10 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
Warming up JVM and Catalyst Optimizer for 15 features...


26/03/23 15:02:17 WARN LAPACK: Failed to load implementation from: com.github.fommil.netlib.NativeSystemLAPACK
26/03/23 15:02:17 WARN LAPACK: Failed to load implementation from: com.github.fommil.netlib.NativeRefLAPACK
26/03/23 15:02:17 WARN BLAS: Failed to load implementation from: com.github.fommil.netlib.NativeSystemBLAS
26/03/23 15:02:17 WARN BLAS: Failed to load implementation from: com.github.fommil.netlib.NativeRefBLAS
26/03/23 15:02:17 WARN InstanceBuilder$NativeBLAS: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/03/23 15:02:17 WARN InstanceBuilder$NativeBLAS: Failed to load implementation from:dev.ludovic.netlib.blas.ForeignLinkerBLAS


26/03/23 15:02:17 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
Warmup complete! Spark is fully optimized for the streaming run.


### Kafka Consumer Configuration

This section initializes the Kafka consumer that will read streaming messages
from the configured topic.

A **unique consumer group ID** is generated using `uuid` so that each execution
of the notebook starts with a fresh consumer session. This avoids conflicts with
previous runs and ensures that the consumer can read messages from the
beginning when necessary.

The configuration also sets `auto.offset.reset="earliest"` so that no data is
missed if the consumer starts after messages have already been produced.

In [ ]:
conf = {
    "bootstrap.servers": "localhost:9092",
    "group.id": f"spotify-gmm-group-{uuid.uuid4()}",
    "auto.offset.reset": "earliest",
    "enable.auto.commit": True
}

consumer = Consumer(conf)
print(f"Kafka consumer created. Topic: {TOPIC_NAME}")

### Streaming and GMM Model Configuration

This section defines the main parameters used for the **streaming Gaussian
Mixture Model (GMM)** pipeline.

The window size determines how frequently incoming Kafka records are grouped
and processed. A fixed random seed is also specified to ensure reproducibility.

Several variables are initialized to store the **persistent state of the GMM
model across streaming windows**, including the cluster means, covariance
matrices, mixture weights, and the effective number of samples assigned to each
component.

In addition, two lists are initialized to track the **AIC (Akaike Information
Criterion)** and **BIC (Bayesian Information Criterion)** values computed at
each window. These metrics help evaluate the quality of the probabilistic
clustering model over time.

In [ ]:
# Time-based window length (seconds) used to group streaming records
WINDOW_SECONDS = 2
# Global random seed used for reproducibility in initialization and warm start
SEED = 42

# Persistent GMM model state across windows
gmm_means = None            # (K, D) component centers
gmm_covariances = None      # (K, D, D) full covariance matrices
gmm_weights = None          # (K,) mixture weights
gmm_effective_n = None      # (K,) effective sample counts per component
global_feature_cols = []    # stable feature order shared by all windows

# Model evaluation metrics per window
aic_history = []
bic_history = []

### Feature Alignment and Dynamic GMM Feature Handling

Before applying the GMM updates, the streaming data must be converted into a
consistent numerical representation.

Since the structure of incoming data may vary between windows, the pipeline
ensures that every batch follows the **same feature order**. Missing columns are
automatically created and filled with zeros so that the feature vectors remain
compatible with the current model, and all feature columns are explicitly cast
to `double` to match the numerical code used in the EM and metric calculations.

Feature alignment is now performed purely in Spark, keeping the data
**distributed across workers** until the exact moment it is consumed by the
Map–Reduce EM update.

If new features appear in later windows, the model structure is automatically
extended: the **mean vectors are padded with zeros**, and the **covariance
matrices are expanded with identity variance** for the new dimensions. This
ensures that the model remains compatible with evolving feature sets in the
streaming data.

In [ ]:
def _align_features_distributed(df, feature_cols):
    """Align incoming window to the global feature layout and numeric types."""
    existing = set(df.columns)
    # Add missing feature columns and cast all features to double
    for c in feature_cols:
        if c not in existing:
            df = df.withColumn(c, F.lit(0.0))
        df = df.withColumn(c, F.col(c).cast("double"))
    return df.na.fill(0.0).select(*feature_cols)

def _extend_gmm_if_new_features(new_feature_cols):
    """
    Extend GMM parameters when unseen features appear in later windows.
    New dimensions get zero-mean and identity covariance.
    """
    global gmm_means, gmm_covariances, global_feature_cols

    if gmm_means is None:
        return

    old_cols = global_feature_cols
    old_set = set(old_cols)

    # Detect new feature columns
    extra = [c for c in new_feature_cols if c not in old_set]
    if not extra:
        return

    num_extra = len(extra)
    K = gmm_means.shape[0]
    D_old = gmm_means.shape[1]
    D_new = D_old + num_extra

    # Expand mean vectors with zeros for the new dimensions
    pad_means = np.zeros((K, num_extra), dtype=float)
    gmm_means = np.hstack([gmm_means, pad_means])

    # Expand covariance matrices with identity blocks on the new dimensions
    new_covs = np.zeros((K, D_new, D_new), dtype=float)
    for k in range(K):
        new_covs[k, :D_old, :D_old] = gmm_covariances[k]
        new_covs[k, D_old:, D_old:] = np.eye(num_extra)
    gmm_covariances = new_covs

    # Record the updated global feature order
    global_feature_cols = old_cols + extra

### Model Selection Metrics: AIC and BIC

This function computes two standard statistical criteria used to evaluate
Gaussian Mixture Models (GMM): **AIC (Akaike Information Criterion)** and
**BIC (Bayesian Information Criterion)**.

Both metrics measure the trade-off between **model fit** and **model
complexity**. They are derived from the distributed log-likelihood of the data
under the current GMM parameters.

The implementation uses a **Map–Reduce pattern**:
- on each worker, a UDF computes the log-probability of each row given the
  current mixture (MAP)
- the log-likelihoods are then summed across the cluster (REDUCE) to obtain
  the total log-likelihood

Lower values indicate a better model because they correspond to a higher
likelihood while penalizing models that use too many parameters. The total
number of parameters includes the cluster means, covariance matrices and
mixture weights.

These metrics are computed at each window to monitor how the probabilistic
clustering model evolves during the streaming process.

In [ ]:
def compute_aic_bic_distributed(df_aligned, means, covariances, weights):
    """Compute AIC/BIC across the cluster using a Map–Reduce style UDF."""
    K, D = means.shape
    N = df_aligned.count()

    # Broadcast current parameters to all workers
    bc_m = spark.sparkContext.broadcast(means)
    bc_c = spark.sparkContext.broadcast(covariances)
    bc_w = spark.sparkContext.broadcast(weights)

    # MAP: Calculate log-probability for each row
    def log_likelihood_udf(*row):

        x = np.array(row)
        p = 0
        for k in range(K):
            reg_cov = bc_c.value[k] + np.eye(D) * 1e-6
            p += bc_w.value[k] * multivariate_normal.pdf(x, mean=bc_m.value[k], cov=reg_cov)
        return float(np.log(max(p, 1e-12)))

    ll_udf = F.udf(log_likelihood_udf, DoubleType())

    # REDUCE: Sum the log-likelihoods over all rows
    total_ll = df_aligned.withColumn("ll", ll_udf(*df_aligned.columns)).agg(F.sum("ll")).collect()[0][0]

    n_params = (K * D) + (K * D * (D + 1) / 2) + (K - 1)
    aic = 2 * n_params - 2 * total_ll
    bic = n_params * np.log(N) - 2 * total_ll
    return aic, bic

### Initial GMM Parameter Estimation

Before performing incremental updates, the Gaussian Mixture Model must be
initialized using the first streaming window.

This function fits a **standard Spark ML GaussianMixture model** on the first
batch of data to estimate the initial parameters of the probabilistic clustering
model. The input features are first converted to numeric values and assembled
into a feature vector compatible with the Spark ML pipeline.

The function extracts the main GMM parameters:
- **means** of each Gaussian component
- **covariance matrices** describing cluster dispersion
- **mixture weights** representing the probability of each component

It also estimates the **effective number of observations per component**, which
is later used when updating the model incrementally as new streaming windows
arrive.

In [ ]:
def initialize_gmm_with_spark(df, k):
    """
    Initialize GMM parameters from the first window using Spark GaussianMixture.
    Returns feature columns, means, covariances, weights, and effective counts.
    """
    feature_cols = df.columns
    if not feature_cols:
        raise ValueError("No features found in the first window.")

    wide = df

    # Cast all feature columns to double and fill missing values with zeros
    for c in feature_cols:
        wide = wide.withColumn(c, col(c).cast("double"))
    wide = wide.na.fill(0.0)

    # Assemble Spark ML features vector
    assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
    data = assembler.transform(wide).select("features")

    # Fit initial Gaussian Mixture model on the first window
    model = GaussianMixture(k=k, seed=SEED, featuresCol="features", predictionCol="prediction").fit(data)

    gaussians = model.gaussians
    centers = np.array([g.mean.toArray() for g in gaussians], dtype=float)
    covs = np.array([g.cov.toArray() for g in gaussians], dtype=float)
    weights = np.array(model.weights, dtype=float)

    # Approximate effective component counts using mixture weights and total N
    N_total = data.count()
    effective_n = weights * N_total

    return feature_cols, centers, covs, weights, effective_n

### Incremental EM Update for the Gaussian Mixture Model

This function performs one **incremental Expectation–Maximization (EM)** update
using the observations contained in the current streaming window.

It is implemented as a true **Map–Reduce** procedure:

#### 1. The E-Step (Distributed MAP Phase)
On each worker, the code computes the *responsibilities* (soft cluster assignments) for every row $x_n$ using the current global means ($\mu_k$), covariances ($\Sigma_k$), and weights ($\pi_k$):

$$
\gamma_{n,k} = \frac{\pi_k \mathcal{N}(x_n | \mu_k, \Sigma_k)}{\sum_{j=1}^K \pi_j \mathcal{N}(x_n | \mu_j, \Sigma_j)}
$$

Each partition then accumulates local sufficient statistics for the current window:
- **Effective count:** $N_k^{(local)} = \sum \gamma_{n,k}$
- **Weighted sum of features:** $\sum \gamma_{n,k} x_n$
- **Weighted outer product (for covariance):** $\sum \gamma_{n,k} (x_n - \mu_k^{old})(x_n - \mu_k^{old})^T$

#### 2. The Aggregation (Distributed REDUCE Phase)
Using `treeAggregate`, these partial statistics are safely summed across the entire Spark cluster to produce the final window-level statistics: $N_k^{(window)}$, $\mu_k^{(window)}$, and $\Sigma_k^{(window)}$.

#### 3. The M-Step (Driver Merge Phase)
On the driver, the global parameters are updated by treating the new window as a weighted running average against the historical data.

First, the algorithm calculates the merging weights based on the number of points seen:
$$w_{old} = \frac{N_k^{(old)}}{N_k^{(old)} + N_k^{(window)}} \quad , \quad w_{new} = \frac{N_k^{(window)}}{N_k^{(old)} + N_k^{(window)}}$$

Then, it updates the global means:
$$
\mu_k^{(new)} = w_{old} \mu_k^{(old)} + w_{new} \mu_k^{(window)}
$$

Next, it updates the covariance matrices. To ensure mathematical accuracy, it includes a third term that corrects the variance to account for the distance between the old mean and the new window's mean:
$$
\Sigma_k^{(new)} = w_{old} \Sigma_k^{(old)} + w_{new} \Sigma_k^{(window)} + w_{old} w_{new} (\mu_k^{(old)} - \mu_k^{(window)})(\mu_k^{(old)} - \mu_k^{(window)})^T
$$

Finally, the global **mixture weights** are updated from the total effective counts across all clusters:
$$
\pi_k^{(new)} = \frac{N_k^{(new)}}{\sum_j N_j^{(new)}}
$$

This incremental design allows the GMM to **adapt continuously to streaming
data without retraining the model from scratch**, while keeping the heavy
numerical work distributed across the Spark cluster.

In [ ]:
def update_gmm_distributed(df_aligned, feature_cols):
    """Distributed EM update using RDD Map–Reduce for full covariance updates."""
    global gmm_means, gmm_covariances, gmm_weights, gmm_effective_n
    K, D = gmm_means.shape

    # Broadcast current global parameters to the workers
    bc_means = spark.sparkContext.broadcast(gmm_means)
    bc_covs = spark.sparkContext.broadcast(gmm_covariances)
    bc_weights = spark.sparkContext.broadcast(gmm_weights)

    # MAP phase: compute responsibilities and partial sufficient statistics
    def process_partition(iterator):


        local_N_k = np.zeros(K)
        local_sum_x = np.zeros((K, D))
        local_sum_cov = np.zeros((K, D, D))

        for row in iterator:
            x = np.array([row[c] for c in feature_cols], dtype=float)
            resp = np.zeros(K)

            for k in range(K):
                reg_cov = bc_covs.value[k] + np.eye(D) * 1e-6
                try:
                    resp[k] = bc_weights.value[k] * multivariate_normal.pdf(x, mean=bc_means.value[k], cov=reg_cov)
                except:
                    resp[k] = 1e-12

            s = resp.sum()
            if s > 0:
                resp /= s

            local_N_k += resp
            for k in range(K):
                local_sum_x[k] += resp[k] * x
                diff = x - bc_means.value[k]
                local_sum_cov[k] += resp[k] * np.outer(diff, diff)

        yield (local_N_k, local_sum_x, local_sum_cov)

    # REDUCE phase: aggregate local statistics from all partitions
    def reduce_partitions(a, b):
        return (a[0] + b[0], a[1] + b[1], a[2] + b[2])

    summary = df_aligned.rdd.mapPartitions(process_partition).treeAggregate(
        (np.zeros(K), np.zeros((K, D)), np.zeros((K, D, D))),
        reduce_partitions,
        reduce_partitions
    )

    # DRIVER: merge window statistics into global GMM parameters
    N_window_k, sum_x, sum_cov = summary
    new_effective_n = gmm_effective_n + N_window_k

    for k in range(K):
        if N_window_k[k] <= 1e-6:
            continue

        # Window-level mean and covariance for component k
        mu_window_k = sum_x[k] / N_window_k[k]
        cov_window_k = sum_cov[k] / N_window_k[k]

        # Weighted merge between previous global parameters and this window
        w_old = gmm_effective_n[k] / new_effective_n[k]
        w_new = N_window_k[k] / new_effective_n[k]

        merged_mean = (w_old * gmm_means[k]) + (w_new * mu_window_k)

        mean_diff = gmm_means[k] - mu_window_k
        diff_outer = np.outer(mean_diff, mean_diff)

        merged_cov = (w_old * gmm_covariances[k]) + (w_new * cov_window_k) + (w_old * w_new * diff_outer)

        gmm_means[k] = merged_mean
        gmm_covariances[k] = merged_cov

    gmm_effective_n = new_effective_n
    gmm_weights = gmm_effective_n / gmm_effective_n.sum()

    return N_window_k

### Streaming Window Processing for Incremental GMM

This function manages the processing of each incoming streaming window for the
incremental **Gaussian Mixture Model**.

For every window, the buffered Kafka records are first converted into a Spark
DataFrame with an explicit numeric schema. The behavior then depends on whether
the model has already been initialized:

- **First window**: the function calls `initialize_gmm_with_spark`, which fits a
  standard Spark `GaussianMixture` model to estimate the initial means,
  covariance matrices, mixture weights, and effective component counts. The
  detected feature columns are stored as the global feature order.

- **Subsequent windows**:
  - `_extend_gmm_if_new_features` checks for any **new feature columns** and
    pads the existing means and covariances so the model can handle the
    expanded feature space.
  - `_align_features_distributed` enforces a consistent feature layout and
    numeric types while keeping the data distributed.
  - `update_gmm_distributed` performs a **distributed EM step** using a
    Map–Reduce pattern over the current window.
  - `compute_aic_bic_distributed` is called on the aligned DataFrame to compute
    **AIC** and **BIC** for the updated model; these values are appended to
    `aic_history` and `bic_history` and logged.

In the current version, only the **post‑update** AIC and BIC are recorded for
each window in order to focus on the final quality of the updated model and
avoid redundant metric computations.

In [10]:
def process_window(records, window_id, k):
    global gmm_means, gmm_covariances, gmm_weights, gmm_effective_n, global_feature_cols

    if not records:
        return

    # Build a numeric schema from the first record and create a Spark DataFrame
    fields = [StructField(c, DoubleType(), True) for c in records[0].keys()]
    schema = StructType(fields)
    df = spark.createDataFrame(records, schema=schema)

    # Use the first window to initialize the GMM with Spark ML
    if gmm_means is None:
        global_feature_cols, gmm_means, gmm_covariances, gmm_weights, gmm_effective_n = initialize_gmm_with_spark(df, k)
        return

    # Extend model parameters if this window introduces new feature columns
    _extend_gmm_if_new_features(df.columns)

    # Align current window to the global feature order in a distributed way
    df_aligned = _align_features_distributed(df, global_feature_cols)

    # Run one distributed EM update of the GMM on this window
    window_soft_counts = update_gmm_distributed(df_aligned, global_feature_cols)

    # Compute and record AIC/BIC after the update for monitoring
    aic_after, bic_after = compute_aic_bic_distributed(df_aligned, gmm_means, gmm_covariances, gmm_weights)
    aic_history.append(aic_after)
    bic_history.append(bic_after)

    print(f"[window {window_id}] AIC: {aic_after:.2f} | BIC: {bic_after:.2f}")

### Kafka Streaming Consumption and Windowed GMM Execution

This function controls the main **Kafka streaming loop** for the incremental
Gaussian Mixture Model.

It continuously consumes messages from the Kafka topic and groups them into
processing windows. A window is processed either when the predefined time limit
is reached or when the buffer size reaches the fast-forward threshold.

Each completed window is passed to the `process_window` function, which performs
the incremental GMM update using the value of **K** specified for the current
experiment.

The function also includes two important mechanisms:
- **benchmark tracking**, used to measure the total execution time for a fixed
  number of windows
- **idle timeout protection**, which ends the experiment early if the stream
  becomes inactive

This logic ensures that the streaming pipeline remains efficient and that the
performance of the incremental GMM can be evaluated across multiple datasets.

In [11]:
def consume_loop_windowed(consumer, topic, target_windows, k):
    consumer.subscribe([topic])
    print(f"Listening on topic: {topic} | Benchmark target: {target_windows} windows | K = {k}")

    records_buffer = []
    window_id = 0
    
    window_start = None 
    first_message_time = None  
    last_message_time = None  
    
    FAST_FORWARD_SIZE = 10000 
    IDLE_TIMEOUT = 10.0       

    try:
        while True:
            now = time.time()

            # Check whether the current window should be processed
            if window_start is not None:
                time_is_up = (now - window_start) >= WINDOW_SECONDS
                buffer_is_full = len(records_buffer) >= FAST_FORWARD_SIZE

                if time_is_up or buffer_is_full:
                    if records_buffer: 
                        window_id += 1
                        if buffer_is_full and not time_is_up:
                            print(f"[Speed Trigger] Fast-forwarding window {window_id} ({len(records_buffer)} rows)")
                            
                        process_window(records_buffer, window_id, k)
                        records_buffer = []
                        
                        # Stop when the benchmark target is reached
                        if window_id == target_windows:
                            total_time = time.time() - first_message_time
                            print(f" BENCHMARK COMPLETE ")
                            print(f"Target: {target_windows} windows")
                            print(f"Total Time: {total_time:.2f} seconds")
                            print(f"Average Time per Window: {total_time / target_windows:.4f} seconds")
                            return total_time
                        
                    window_start = time.time()

            # Exit early if the stream becomes inactive
            if last_message_time is not None and (now - last_message_time) > IDLE_TIMEOUT:
                print(f" Stream went quiet for {IDLE_TIMEOUT} seconds. Ending early at window {window_id}.")
                total_time = (time.time() - first_message_time) - IDLE_TIMEOUT 
                print(f" BENCHMARK COMPLETE (EARLY EXIT) ")
                print(f"Target: {target_windows} windows | Completed: {window_id} windows")
                print(f"Total Time: {total_time:.2f} seconds")
                return total_time

            # Poll Kafka for new messages
            msg = consumer.poll(timeout=0.1)

            if msg is None:
                continue
            
            last_message_time = time.time()

            if msg.error():
                if msg.error().code() == KafkaError._PARTITION_EOF:
                    continue
                print("Kafka error:", msg.error())
                continue
            
            # Initialize timers on first message
            if window_start is None:
                window_start = time.time()
                first_message_time = window_start
                last_message_time = window_start 
                print("First message received! Starting the benchmark timer...")

            # Parse message and add it to the current buffer
            try:
                payload = msg.value().decode("utf-8")
                obj = json.loads(payload)
            except Exception as e:
                print("Bad JSON message, skipping. Error:", e)
                continue

            if isinstance(obj, dict):
                records_buffer.append(obj)
            elif isinstance(obj, list):
                records_buffer.extend([x for x in obj if isinstance(x, dict)])
            else:
                continue

    except KeyboardInterrupt:
        print("\nStopping (KeyboardInterrupt).")
        if first_message_time:
             return time.time() - first_message_time
        return 0

    finally:
        # Make sure the last partial window is processed before shutting down
        if records_buffer and window_id < target_windows:
            window_id += 1
            process_window(records_buffer, window_id, k)
        print("Finished consuming current dataset.")

### GMM Model State Reset

This utility function resets the global state of the **Gaussian Mixture Model**
before starting a new streaming experiment.

All variables storing the model parameters and evaluation history are cleared,
including the component means, covariance matrices, mixture weights, and
effective sample counts. The stored **AIC and BIC histories** are also reset.

This ensures that each dataset run starts from a **clean model initialization**
and that results from previous experiments do not influence the next one.

In [12]:
def reset_model_state():
    """Reset all global variables storing the GMM model state."""
    global gmm_means, gmm_covariances, gmm_weights, gmm_effective_n
    global global_feature_cols, aic_history, bic_history, last_window_soft_counts
    
    gmm_means = None
    gmm_covariances = None
    gmm_weights = None
    gmm_effective_n = None
    global_feature_cols = []
    aic_history = []
    bic_history = []
    last_window_soft_counts = None
    print(" GMM Model state has been completely reset.")

### Global Evaluation of the Final GMM Model (AIC / BIC)

After the streaming experiment finishes, this function evaluates the final
Gaussian Mixture Model using the **entire dataset**.

The dataset is reloaded from the exported CSV file that was used during
streaming. All features are aligned with the global feature structure used by
the model so that the data representation remains consistent.

Using the final learned parameters (means, covariance matrices, and mixture
weights), the function computes the **global AIC and BIC scores** using the
same **distributed Map–Reduce** logic that was applied per window.

The function also prints a short summary including the number of model
parameters and the final AIC and BIC scores, so the global model quality can be
compared across experiments.

In [13]:
def evaluate_full_dataset_aic_bic(spark, final_means, final_covariances, final_weights, feature_cols, csv_path):
    """
    Load the full dataset and compute global AIC and BIC using 
    the same distributed Map–Reduce logic used per window.
    """
    if final_means is None:
        print("Model has not been trained. No GMM parameters found.")
        return None, None

    print(f"Loading data for distributed evaluation from: {csv_path} ...")
    # Load full CSV and immediately align its columns to the global feature order
    df_full = spark.read.csv(csv_path, header=True, inferSchema=True)
    df_aligned = _align_features_distributed(df_full, feature_cols)

    # Reuse the distributed metric function for a global evaluation pass
    global_aic, global_bic = compute_aic_bic_distributed(
        df_aligned, final_means, final_covariances, final_weights
    )

    K, D = final_means.shape
    n_params = (K * D) + (K * D * (D + 1) / 2) + (K - 1)

    print("-" * 40)
    print(f"GLOBAL DISTRIBUTED GMM EVALUATION")
    print(f"File: {csv_path}")
    print(f"Total parameters (k): {int(n_params)}")
    print(f"Final Global AIC Score: {global_aic:.2f}")
    print(f"Final Global BIC Score: {global_bic:.2f}")
    print("-" * 40)

    return global_aic, global_bic

### Experimental Execution and Benchmarking for Incremental GMM

This section runs the complete streaming Gaussian Mixture Model pipeline on
multiple datasets in order to evaluate its performance and model quality.

Each experiment defines:
- the dataset file
- the total number of rows
- the batch size used during streaming
- the value of **K** selected for that dataset

For every dataset, the pipeline performs the following steps:
1. resets the GMM model state
2. consumes and processes the streaming windows through Kafka
3. evaluates the final probabilistic clustering model on the full dataset using
   **global AIC and BIC** computed in a distributed fashion
4. stores the execution time and evaluation metrics in a summary table

The resulting table is displayed at the end of the notebook using `pandas` and
IPython’s display utilities.

This experimental loop makes it possible to compare the behavior of the
incremental GMM across datasets of different sizes and with different numbers
of Gaussian components.

In [14]:

# Define the datasets and GMM configuration used in the experiments
experiments = [
    {"file": "exported_dataframes/df_30k_part_1.csv", "rows": 30000, "batch_size": 10000, "k": 14},
    {"file": "exported_dataframes/df_30k_part_2.csv", "rows": 30000, "batch_size": 10000, "k": 15},
    {"file": "exported_dataframes/df_30k_part_3.csv", "rows": 30000, "batch_size": 10000, "k": 14},
    {"file": "exported_dataframes/df_60k_part_1.csv", "rows": 60000, "batch_size": 10000, "k": 15},
    {"file": "exported_dataframes/df_60k_part_2.csv", "rows": 60000, "batch_size": 10000, "k": 15},
    {"file": "exported_dataframes/df_60k_part_3.csv", "rows": 60000, "batch_size": 10000, "k": 15},
    {"file": "exported_dataframes/df_90k_part_1.csv", "rows": 90000, "batch_size": 10000, "k": 14},
    {"file": "exported_dataframes/df_90k_part_2.csv", "rows": 90000, "batch_size": 10000, "k": 15},
    {"file": "exported_dataframes/df_90k_part_3.csv", "rows": 90000, "batch_size": 10000, "k": 12},
    {"file": "exported_dataframes/spotify_data_prepped.csv", "rows": 113999, "batch_size": 10000, "k": 15} 
]

TOPIC_NAME = "final_run_000"
experiment_results = []

for exp in experiments:
    file_path = exp["file"]
    
    target_windows = math.ceil(exp["rows"] / exp["batch_size"])
    k_value = exp["k"]
    
    print("\n" + "=" * 70)
    print(f" STARTING EXPERIMENT: {file_path} ({exp['rows']} rows) | K = {k_value}")
    print("=" * 70)
    
    # Reset the GMM model before each experiment so runs are independent
    reset_model_state()
    
    # Run the streaming consumer and benchmark execution time
    total_time = consume_loop_windowed(consumer, TOPIC_NAME, target_windows, k_value)
    
    # Evaluate Global AIC & BIC using the finalized GMM model on the full dataset
    global_aic, global_bic = evaluate_full_dataset_aic_bic(
        spark, gmm_means, gmm_covariances, gmm_weights, global_feature_cols, file_path
    )
    
    # Store experiment results
    experiment_results.append({
        "Dataset": file_path,
        "Rows": exp["rows"],
        "K": k_value,
        "Target Windows": target_windows,
        "Total Time (sec)": round(total_time, 2),
        "Global AIC": round(global_aic, 2),
        "Global BIC": round(global_bic, 2)
    })

# Display final summary table
print("\n ALL EXPERIMENTS COMPLETE ")
results_df = pd.DataFrame(experiment_results)
display.display(results_df)

# Close the Kafka consumer after all experiments
consumer.close()
print("Kafka consumer permanently closed.")


 STARTING EXPERIMENT: exported_dataframes/df_30k_part_1.csv (30000 rows) | K = 14
 GMM Model state has been completely reset.
Listening on topic: final_run_000 | Benchmark target: 3 windows | K = 14
First message received! Starting the benchmark timer...
[Speed Trigger] Fast-forwarding window 2 (10000 rows)


[window 2] AIC: -307668.27 | BIC: -293946.99
[Speed Trigger] Fast-forwarding window 3 (10000 rows)


[window 3] AIC: -306131.29 | BIC: -292410.01
 BENCHMARK COMPLETE 
Target: 3 windows
Total Time: 24.84 seconds
Average Time per Window: 8.2800 seconds
Finished consuming current dataset.
Loading data for distributed evaluation from: exported_dataframes/df_30k_part_1.csv ...


----------------------------------------
GLOBAL DISTRIBUTED GMM EVALUATION
File: exported_dataframes/df_30k_part_1.csv
Total parameters (k): 1903
Final Global AIC Score: -929411.44
Final Global BIC Score: -913599.50
----------------------------------------

 STARTING EXPERIMENT: exported_dataframes/df_30k_part_2.csv (30000 rows) | K = 15
 GMM Model state has been completely reset.
Listening on topic: final_run_000 | Benchmark target: 3 windows | K = 15
First message received! Starting the benchmark timer...
[Speed Trigger] Fast-forwarding window 1 (10000 rows)
[Speed Trigger] Fast-forwarding window 2 (10000 rows)


[window 2] AIC: -279076.86 | BIC: -264374.97
[Speed Trigger] Fast-forwarding window 3 (10000 rows)


...

[window 11] AIC: -342263.71 | BIC: -327561.82


[window 12] AIC: -228407.22 | BIC: -214432.89
 BENCHMARK COMPLETE 
Target: 12 windows
Total Time: 64.76 seconds
Average Time per Window: 5.3965 seconds
Finished consuming current dataset.
Loading data for distributed evaluation from: exported_dataframes/spotify_data_prepped.csv ...


----------------------------------------
GLOBAL DISTRIBUTED GMM EVALUATION
File: exported_dataframes/spotify_data_prepped.csv
Total parameters (k): 2039
Final Global AIC Score: -3966092.87
Final Global BIC Score: -3946428.87
----------------------------------------

 ALL EXPERIMENTS COMPLETE 


,Dataset,Rows,K,Target Windows,Total Time (sec),Global AIC,Global BIC
0,exported_dataframes/df_30k_part_1.csv,30000,14,3,24.84,-929411.44,-913599.50
1,exported_dataframes/df_30k_part_2.csv,30000,15,3,18.33,-857743.92,-840801.97
2,exported_dataframes/df_30k_part_3.csv,30000,14,3,17.55,-838561.57,-822749.64
3,exported_dataframes/df_60k_part_1.csv,60000,15,6,33.20,-1863147.45,-1844792.17
4,exported_dataframes/df_60k_part_2.csv,60000,15,6,33.18,-1914051.70,-1895696.42
5,exported_dataframes/df_60k_part_3.csv,60000,15,6,31.98,-2235246.93,-2216891.65
6,exported_dataframes/df_90k_part_1.csv,90000,14,9,46.58,-2744949.56,-2727046.97
7,exported_dataframes/df_90k_part_2.csv,90000,15,9,47.34,-2731468.56,-2712286.54
8,exported_dataframes/df_90k_part_3.csv,90000,12,9,41.99,-2759941.54,-2744597.80
9,exported_dataframes/spotify_data_prepped.csv,113999,15,12,64.76,-3966092.87,-3946428.87


Kafka consumer permanently closed.
